In [1]:
import sys 
from pathlib import Path 

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root)) 

from src.evaluate_models import eval_reg

print("Imports successful")

Imports successful


In [2]:
import pandas as pd

df = pd.read_csv(
    "../data/student-mat.csv",
    sep=";"
)

df_encoded = pd.get_dummies(
    df,
    drop_first=True
)

X_full = df_encoded.drop(
    columns=["G3"]
).copy()

y = df_encoded["G3"].copy()

print("Dataset loaded.")
print("X_full shape:", X_full.shape)
print("y shape:", y.shape)

Dataset loaded.
X_full shape: (395, 41)
y shape: (395,)


In [3]:
from sklearn.model_selection import train_test_split 

Xtr_f, Xte_f, ytr, yte = train_test_split(
    X_full,
    y,
    test_size=0.2,
    random_state=42
)

print("Train-test split completed.")
print("Xtr_f:", Xtr_f.shape)
print("Xte_f:", Xte_f.shape)


Train-test split completed.
Xtr_f: (316, 41)
Xte_f: (79, 41)


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

print("Session 26 imports successful")

Session 26 imports successful


In [5]:
session26_models = {
    "KNN": Pipeline([
        ("standardscaler", StandardScaler()),
        ("kneighborsregressor", KNeighborsRegressor())
    ]),
    "SVR": Pipeline([
        ("standardscaler", StandardScaler()),
        ("svr", SVR())
    ])
}

print("Pipelines created.")

Pipelines created.


In [6]:
for model_name, pipeline in session26_models.items():

    pipeline.fit(
        Xtr_f,
        ytr
    )

    print(f"{model_name} fitted successfully.")

KNN fitted successfully.
SVR fitted successfully.


In [7]:
for model_name, pipeline in session26_models.items():

    print(f"\n{model_name}")

    for step_number, (
        step_name,
        step_object
    ) in enumerate(
        pipeline.steps,
        start=1
    ):
        print(
            f"Step {step_number}: "
            f"{step_name} -> {step_object}"
        )


KNN
Step 1: standardscaler -> StandardScaler()
Step 2: kneighborsregressor -> KNeighborsRegressor()

SVR
Step 1: standardscaler -> StandardScaler()
Step 2: svr -> SVR()


In [8]:
from sklearn.preprocessing import StandardScaler

for model_name, pipeline in session26_models.items():

    first_step_name, first_step_object = pipeline.steps[0]

    assert isinstance(
        first_step_object,
        StandardScaler
    ), (
        f"{model_name} does not have "
        "StandardScaler as its first step."
    )

    print(
        f"{model_name}: scaling is correctly "
        "placed before the estimator."
    )

KNN: scaling is correctly placed before the estimator.
SVR: scaling is correctly placed before the estimator.


In [10]:
import pandas as pd
import numpy as np

numeric_Xtr_f = Xtr_f.astype(float)

feature_ranges = pd.DataFrame({
    "Minimum": numeric_Xtr_f.min(),
    "Maximum": numeric_Xtr_f.max(),
    "Mean": numeric_Xtr_f.mean(),
    "Standard_Deviation": numeric_Xtr_f.std()
})

feature_ranges["Range"] = (
    feature_ranges["Maximum"]
    - feature_ranges["Minimum"]
)

feature_ranges = feature_ranges.sort_values(
    by="Range",
    ascending=False
)

feature_ranges.head(15)

,Minimum,Maximum,Mean,Standard_Deviation,Range
absences,0.0,75.0,5.905063,8.422033,75.0
G2,0.0,19.0,10.651899,3.755930,19.0
G1,5.0,19.0,10.933544,3.216823,14.0
age,15.0,22.0,16.743671,1.270217,7.0
Medu,0.0,4.0,2.734177,1.080375,4.0
famrel,1.0,5.0,3.943038,0.885464,4.0
freetime,1.0,5.0,3.218354,1.020323,4.0
goout,1.0,5.0,3.161392,1.119480,4.0
Dalc,1.0,5.0,1.500000,0.903257,4.0
Walc,1.0,5.0,2.344937,1.296395,4.0


In [11]:
knn_scaler = session26_models["KNN"].named_steps["standardscaler"]
svr_scaler = session26_models["SVR"].named_steps["standardscaler"]

print("KNN scaler:", knn_scaler)
print("SVR scaler:", svr_scaler)

KNN scaler: StandardScaler()
SVR scaler: StandardScaler()


In [12]:
print("First five KNN scaler means:")
print(knn_scaler.mean_[:5])

print("\nFirst five KNN scaler standard deviations:")
print(knn_scaler.scale_[:5])

First five KNN scaler means:
[16.74367089  2.73417722  2.5443038   1.43037975  2.04746835]

First five KNN scaler standard deviations:
[1.26820583 1.07866374 1.07676812 0.68775077 0.83493344]


In [13]:
Xtr_scaled_demo = knn_scaler.transform(Xtr_f)

print("Scaled training shape:", Xtr_scaled_demo.shape)

print("\nFirst three scaled rows:")
print(Xtr_scaled_demo[:3])

Scaled training shape: (316, 41)

First three scaled rows:
[[-0.58639605  0.24643712  0.42320737 -0.62577865 -0.05685286 -0.45674383
   0.06443217 -1.19598096 -0.14439599 -0.55443033 -0.2664957  -0.36847328
  -0.46440769  0.33205033  0.62616324 -0.34722813  1.07906606  0.52853782
  -0.63105474  0.33567254 -0.30565602 -0.73073975  1.66189794 -0.40219983
  -0.23094011  0.88050037 -0.59684435 -0.26680787  1.66189794 -0.33567254
  -0.59684435  0.66552414 -0.30565602 -0.40219983 -1.23612297  1.08596056
   1.00634927  0.49901088  0.24576958  0.43869446  1.37807205]
 [-0.58639605 -0.68063585  0.42320737  0.82823645 -1.25455313 -0.45674383
   1.19557467 -0.21434464 -0.14439599 -0.55443033 -1.03908875 -0.36847328
  -0.70225668  0.64340909  0.89283114 -0.34722813  1.07906606  0.52853782
  -0.63105474  0.33567254 -0.30565602  1.36847626 -0.60172167 -0.40219983
  -0.23094011  0.88050037 -0.59684435 -0.26680787  1.66189794 -0.33567254
  -0.59684435 -1.50257511 -0.30565602 -0.40219983 -1.23612297 -0

In [14]:
import pandas as pd
import numpy as np

scaled_summary = pd.DataFrame({
    "Scaled_Mean": np.mean(
        Xtr_scaled_demo,
        axis=0
    ),
    "Scaled_Std": np.std(
        Xtr_scaled_demo,
        axis=0
    )
})

scaled_summary.head(10)

,Scaled_Mean,Scaled_Std
0,-6.928354e-16,1.0
1,1.054009e-16,1.0
2,1.911270e-16,1.0
3,-1.798842e-16,1.0
4,2.670157e-16,1.0
5,3.091760e-17,1.0
6,-1.939377e-16,1.0
7,1.686415e-16,1.0
8,3.513364e-17,1.0
9,0.000000e+00,1.0


In [15]:
feature_index = 0

feature_name = Xtr_f.columns[feature_index]

original_values = Xtr_f.iloc[
    :10,
    feature_index
].to_numpy()

scaled_values = Xtr_scaled_demo[
    :10,
    feature_index
]

before_after = pd.DataFrame({
    "Original_Value": original_values,
    "Scaled_Value": scaled_values
})

print("Feature:", feature_name)

before_after

Feature: age


,Original_Value,Scaled_Value
0,16,-0.586396
1,16,-0.586396
2,16,-0.586396
3,16,-0.586396
4,16,-0.586396
5,18,0.990635
6,17,0.202119
7,19,1.779151
8,17,0.202119
9,18,0.990635


In [16]:
knn_predictions = session26_models[
    "KNN"
].predict(Xte_f)

svr_predictions = session26_models[
    "SVR"
].predict(Xte_f)

print("KNN predictions:", len(knn_predictions))
print("SVR predictions:", len(svr_predictions))
print("Test observations:", len(yte))

KNN predictions: 79
SVR predictions: 79
Test observations: 79


In [17]:
knn_metrics = eval_reg(
    yte,
    knn_predictions
)

svr_metrics = eval_reg(
    yte,
    svr_predictions
)

print("KNN metrics:", knn_metrics)
print("SVR metrics:", svr_metrics)

KNN metrics: {'MAE': 2.5848101265822785, 'RMSE': 3.3926950565642415, 'R2': 0.4386562685587473}
SVR metrics: {'MAE': 1.8366678536589514, 'RMSE': 2.7260297218073055, 'R2': 0.6375898115704413}


In [18]:
import pandas as pd

session26_results = pd.DataFrame([
    {
        "Model": "KNN",
        "MAE": knn_metrics["MAE"],
        "RMSE": knn_metrics["RMSE"],
        "R2": knn_metrics["R2"],
    },
    {
        "Model": "SVR",
        "MAE": svr_metrics["MAE"],
        "RMSE": svr_metrics["RMSE"],
        "R2": svr_metrics["R2"],
    },
])

session26_results.round(4)

,Model,MAE,RMSE,R2
0,KNN,2.5848,3.3927,0.4387
1,SVR,1.8367,2.7260,0.6376
